<a href="https://colab.research.google.com/github/Thanaphon-673020253-2/project_bigdata/blob/main/PART3_FINAL.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

---
# Part 3: Data Collection Robot + External Business Insight *(งานกลุ่ม — 25 คะแนน)*

ส่วนนี้ให้ทีมลอง **สร้าง dataset เล็ก ๆ ของตัวเอง** จากข้อมูลออนไลน์ แล้วใช้ตอบคำถามหนึ่งข้อ

เลือกวิธีเก็บเพียง **1 วิธี**
* API
* RSS
* Web Robot / Crawler / Scraper

## สิ่งที่ต้องทำ

1. ตั้งคำถามว่า **ข้อมูลนี้จะช่วยตัดสินใจเรื่องอะไร**
2. นิยามก่อนว่า **1 record คืออะไร** เช่น 1 ข่าว / 1 post / 1 complaint
3. เก็บอย่างน้อย **200 valid analytical records หลัง clean**
4. ตรวจ duplicate, missing และ record ที่ใช้ไม่ได้
5. ใช้เทคนิควิเคราะห์ที่เหมาะสม **1–2 เทคนิคก็เพียงพอ**
6. สรุป **1 Insight Card** พร้อมข้อจำกัด

**ข้อกำหนดด้านความรับผิดชอบ**
* เคารพ `robots.txt`, rate limit และ Terms of Service
* ใส่ delay เมื่อดึงหน้าเว็บ
* ห้าม bypass login, CAPTCHA หรือ paywall
* ไม่เก็บข้อมูลส่วนบุคคลที่ไม่จำเป็น

> Part 3 ไม่ได้วัดว่าใคร scrape ได้เยอะที่สุด แต่วัดว่าใครสร้างข้อมูลที่ใช้วิเคราะห์ได้จริงและตอบคำถามได้


### 3.1 เลือกแหล่งข้อมูลและวิธีเก็บ

เลือกแหล่งข้อมูลที่สัมพันธ์กับคำถามของทีม แล้วใช้ **เพียง 1 วิธีเก็บ**

ตัวอย่าง:
* Hacker News API — post + score + comments
* RSS — ข่าว/บทความ
* Traffy Fondue Open Data — เรื่องร้องเรียน
* เว็บไซต์สาธารณะที่อนุญาต crawler — ข่าว/บทความ/รายการต่าง ๆ

**กรอกก่อนเขียนโค้ด**
* แหล่งข้อมูล: ...
* คำถาม: ...
* ผู้ใช้ผลวิเคราะห์: ...
* **1 record คือ:** ...
* วิธีเก็บ: API / RSS / Robot
* field สำคัญที่จะเก็บ: ...

ด้านล่างมี
* **ตัวอย่างที่ 1:** API แบบสั้น
* **ตัวอย่างที่ 2–3:** ทางเลือกอื่นแบบสรุป
* **ตัวอย่างที่ 4:** Web Robot แบบอธิบายทีละขั้น


* # Source : The Movie DB
* # Question :
* # Person Analytic : platform streaming
* # 1 record  : 1 review


# **3.1 เตรียมข้อมูล**
* # ไม่ต้องรันแล้ว ใช้ไฟล์ที่โหลดเอานะ

In [1]:
# ===== Setup =====
import os, json, time
from getpass import getpass
import numpy as np
import pandas as pd
import requests

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)

# ---- โฟลเดอร์และไฟล์ ----
DATA_DIR      = "./data_part3/"
os.makedirs(DATA_DIR, exist_ok=True)
TV_RAW_PATH   = DATA_DIR + "tmdb_tv_raw.jsonl"
REV_RAW_PATH  = DATA_DIR + "tmdb_reviews_raw.jsonl"
LOG_PATH      = DATA_DIR + "collection_log.json"
CORPUS_JSONL  = DATA_DIR + "tmdb_corpus_clean.jsonl"
CORPUS_CSV    = DATA_DIR + "tmdb_corpus_clean.csv"

# ---- สวิตช์: False = ถ้ามีไฟล์ raw อยู่แล้วให้ใช้ไฟล์เดิม (ข้อมูลไม่เปลี่ยน) ----
FORCE_REFETCH = False

# ---- พารามิเตอร์การเก็บข้อมูล ----
LANGUAGE         = "en-US"
SORT_BY          = "popularity.desc"
MAX_PAGES        = 30        # 1 หน้า = 20 เรื่อง -> ~600 เรื่อง
MAX_REVIEW_PAGES = 30        # 1 หน้า <= 20 รีวิว/เรื่อง (หยุดเองเมื่อหมดหน้า)
DELAY            = 0.1       # วินาที ระหว่าง request

print("ข้อมูลจะถูกบันทึกที่:", os.path.abspath(DATA_DIR))

ข้อมูลจะถูกบันทึกที่: /content/data_part3


# ฟังก์ชันที่ใช้เก็บข้อมูล
ยังไม่เรียก API ในเซลล์นี้ (token จะถูกขอเมื่อมีการเรียกครั้งแรกเท่านั้น)

In [2]:
# ===== A0: ฟังก์ชัน =====
BASE_URL = "https://api.themoviedb.org/3"
session = None

def get_session():
    """ใช้ "API Read Access Token" (ขึ้นต้น eyJ...) ไม่ใช่ API Key 32 ตัว
    Colab: เก็บเป็น Secret ชื่อ TMDB_TOKEN | ที่อื่น: ตัวแปรสภาพแวดล้อม TMDB_TOKEN หรือพิมพ์ตอนรัน
    ห้ามเขียน token ลงใน notebook"""
    try:
        from google.colab import userdata
        token = userdata.get("TMDB_TOKEN")
    except Exception:
        token = os.environ.get("TMDB_TOKEN") or getpass("วาง TMDB API Read Access Token: ")
    s = requests.Session()
    s.headers.update({"Authorization": f"Bearer {token}", "accept": "application/json"})
    return s

def tmdb_get(path, params=None, retries=4):
    """เรียก TMDB พร้อม retry เมื่อโดน rate limit (429) หรือ server error (5xx)"""
    global session
    if session is None:
        session = get_session()
    for attempt in range(retries):
        r = session.get(BASE_URL + path, params=params, timeout=20)
        if r.status_code == 429:
            wait = int(r.headers.get("Retry-After", 2))
            print(f"โดน rate limit รอ {wait}s ...")
            time.sleep(wait)
            continue
        if r.status_code >= 500:
            time.sleep(2 * (attempt + 1))
            continue
        r.raise_for_status()
        return r.json()
    raise RuntimeError(f"เรียก {path} ไม่สำเร็จหลัง {retries} ครั้ง")

def load_jsonl(path):
    """อ่าน JSON Lines ทีละบรรทัด (ถ้าไฟล์เสียจะบอกเลขบรรทัด)"""
    rows = []
    with open(path, encoding="utf-8") as f:
        for n, line in enumerate(f, 1):
            if line.strip():
                try:
                    rows.append(json.loads(line))
                except json.JSONDecodeError as e:
                    raise ValueError(f"{path} บรรทัด {n} อ่านไม่ได้: {e}")
    return pd.DataFrame(rows)

def save_jsonl(df, path):
    df.to_json(path, orient="records", lines=True, force_ascii=False, date_format="iso")

def load_log():
    return json.load(open(LOG_PATH, encoding="utf-8")) if os.path.exists(LOG_PATH) else {}

def save_log(log):
    json.dump(log, open(LOG_PATH, "w", encoding="utf-8"), ensure_ascii=False, indent=1)

# เก็บซีรีส์ (/discover/tv → /tv/{id})
ถ้ามี tmdb_tv_raw.jsonl อยู่แล้วและ FORCE_REFETCH = False จะอ่านไฟล์เดิม ไม่เรียก API

In [3]:
# ===== A1: ซีรีส์ =====
def fetch_discover(max_pages=MAX_PAGES):
    rows = []
    for page in range(1, max_pages + 1):
        data = tmdb_get("/discover/tv", {"language": LANGUAGE, "sort_by": SORT_BY,
                                         "include_adult": "false", "page": page})
        results = data.get("results", [])
        if not results:
            break
        rows.extend(results)
        if page % 10 == 0:
            print(f"ดึงรายการแล้ว {page} หน้า / {len(rows)} เรื่อง")
        if page >= data.get("total_pages", page):
            break
        time.sleep(DELAY)
    return rows

def build_tv_record(d, item, genre_map):
    tv_id = d["id"]
    return {
        "doc_id": str(tv_id),
        "name": d.get("name"),
        "original_name": d.get("original_name"),
        "original_language": d.get("original_language"),
        "origin_country": "|".join(d.get("origin_country") or []),
        "first_air_date": d.get("first_air_date"),
        "last_air_date": d.get("last_air_date"),
        "genres": "|".join(g["name"] for g in d.get("genres", [])) or "|".join(genre_map.get(x, str(x)) for x in item.get("genre_ids", [])),
        "overview": d.get("overview") or item.get("overview") or "",
        "vote_average": d.get("vote_average"),
        "vote_count": d.get("vote_count"),
        "popularity": d.get("popularity"),
        "number_of_seasons": d.get("number_of_seasons"),
        "number_of_episodes": d.get("number_of_episodes"),
        "status": d.get("status"),
        "in_production": d.get("in_production"),
        "source": "tmdb",
        "url": f"https://www.themoviedb.org/tv/{tv_id}",
    }

if os.path.exists(TV_RAW_PATH) and not FORCE_REFETCH:
    print("พบ", TV_RAW_PATH, "-> ใช้ไฟล์เดิม ไม่เรียก API")
    df_tv_raw = load_jsonl(TV_RAW_PATH)
    df_tv_raw["doc_id"] = df_tv_raw["doc_id"].astype(str)
    log = load_log()
else:
    genre_map = {g["id"]: g["name"] for g in tmdb_get("/genre/tv/list", {"language": LANGUAGE})["genres"]}
    list_rows = fetch_discover()

    seen, uniq = set(), []                         # /discover/tv อาจส่งซีรีส์เดิมซ้ำข้ามหน้า -> ตัดก่อนเรียก detail
    for it in list_rows:
        if it["id"] not in seen:
            seen.add(it["id"]); uniq.append(it)

    records, failed = [], []
    for i, item in enumerate(uniq, start=1):
        try:
            records.append(build_tv_record(tmdb_get(f"/tv/{item['id']}", {"language": LANGUAGE}), item, genre_map))
        except Exception as e:
            failed.append((item["id"], str(e)))
        if i % 100 == 0:
            print(f"detail {i}/{len(uniq)}")
        time.sleep(DELAY)

    df_tv_raw = pd.DataFrame(records)
    save_jsonl(df_tv_raw, TV_RAW_PATH)
    log = {"series_listed": len(list_rows), "series_list_duplicates": len(list_rows) - len(uniq),
           "series_detail_ok": len(records), "series_detail_failed": len(failed)}
    save_log(log)
    print(f"✅ ซีรีส์: รายการ {len(list_rows)} -> ไม่ซ้ำ {len(uniq)} -> detail สำเร็จ {len(records)} / ล้มเหลว {len(failed)}")

print("ซีรีส์ดิบ:", len(df_tv_raw), "เรื่อง")

ดึงรายการแล้ว 10 หน้า / 200 เรื่อง
ดึงรายการแล้ว 20 หน้า / 400 เรื่อง
ดึงรายการแล้ว 30 หน้า / 600 เรื่อง
detail 100/596
detail 200/596
detail 300/596
detail 400/596
detail 500/596
✅ ซีรีส์: รายการ 600 -> ไม่ซ้ำ 596 -> detail สำเร็จ 596 / ล้มเหลว 0
ซีรีส์ดิบ: 596 เรื่อง


# เก็บรีวิว (/tv/{id}/reviews)
วนทีละซีรีส์ ไม่ซ้ำ id (ไม่วนตามตารางที่มี id ซ้ำ) และไม่เก็บชื่อผู้รีวิว

In [4]:
MAX_REVIEW_PAGES_LOCAL = MAX_REVIEW_PAGES

def fetch_reviews(tv_id, max_pages=MAX_REVIEW_PAGES_LOCAL):
    out = []
    for page in range(1, max_pages + 1):
        data = tmdb_get(f"/tv/{tv_id}/reviews", {"page": page})
        for r in data.get("results", []):
            out.append({
                "review_id": r.get("id"),
                "doc_id": str(tv_id),
                "text": r.get("content") or "",
                "user_rating": (r.get("author_details") or {}).get("rating"),   # อาจเป็น None
                "created_at": r.get("created_at"),
                # ไม่เก็บ author/username เพื่อลดข้อมูลส่วนบุคคล (PDPA)
            })
        if page >= data.get("total_pages", 1):
            break
        time.sleep(DELAY)
    return out

if os.path.exists(REV_RAW_PATH) and not FORCE_REFETCH:
    print("พบ", REV_RAW_PATH, "-> ใช้ไฟล์เดิม ไม่เรียก API")
    df_rev_raw = load_jsonl(REV_RAW_PATH)
    df_rev_raw["doc_id"] = df_rev_raw["doc_id"].astype(str)
else:
    ids = df_tv_raw["doc_id"].drop_duplicates().tolist()
    rev_rows, rev_failed = [], []
    for i, tv_id in enumerate(ids, start=1):
        try:
            rev_rows.extend(fetch_reviews(tv_id))
        except Exception as e:
            rev_failed.append(tv_id)
        if i % 100 == 0:
            print(f"{i}/{len(ids)} เรื่อง | รีวิวสะสม {len(rev_rows)}")
        time.sleep(DELAY)

    n_first_fail = len(rev_failed)                  # ลองใหม่อีก 1 รอบสำหรับเรื่องที่ล้มเหลว
    still_failed = []
    for tv_id in rev_failed:
        try:
            rev_rows.extend(fetch_reviews(tv_id))
        except Exception:
            still_failed.append(tv_id)
        time.sleep(DELAY)

    df_rev_raw = pd.DataFrame(rev_rows)
    save_jsonl(df_rev_raw, REV_RAW_PATH)
    log.update({"review_series_requested": len(ids), "review_series_failed_first_try": n_first_fail,
                "review_series_failed_final": len(still_failed), "review_rows_raw": len(df_rev_raw)})
    save_log(log)
    print(f"✅ รีวิว: ขอจาก {len(ids)} เรื่อง -> ได้ {len(df_rev_raw)} แถว | ล้มเหลวรอบแรก {n_first_fail} / หลังลองใหม่ {len(still_failed)}")

print("รีวิวดิบ:", len(df_rev_raw), "แถว")

100/596 เรื่อง | รีวิวสะสม 144
200/596 เรื่อง | รีวิวสะสม 273
300/596 เรื่อง | รีวิวสะสม 335
400/596 เรื่อง | รีวิวสะสม 408
500/596 เรื่อง | รีวิวสะสม 490
✅ รีวิว: ขอจาก 596 เรื่อง -> ได้ 576 แถว | ล้มเหลวรอบแรก 0 / หลังลองใหม่ 0
รีวิวดิบ: 576 แถว


# --------------------------**ห้ามรันจนถึงส่วนนี้น้า** ---------------------------

# **3.2 ตรวจคุณภาพข้อมูล (Clean + Quality Report)**
ลำดับ: Clean ซีรีส์ → Clean รีวิว → Join → รายงาน → บันทึก final corpus
ทุกเซลล์ในส่วนนี้ใช้เฉพาะไฟล์ raw ที่โหลดแล้ว จึงรันซ้ำได้โดยผลเดิม